In [43]:
import numpy as np
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve
import numpy as np
from xgboost import XGBClassifier

In [44]:
data1 = np.load("E:/Pythonfile/Facial-Recognition/data/feature/casia_lbp_face_u_hsv_features.npz")
X1_train = data1["X_train"]
y1_train = data1["y_train"]
X1_test = data1["X_test"]
y1_test = data1["y_test"]

data2 = np.load("E:/Pythonfile/Facial-Recognition/data/feature/casia_lbp_face_u_ycrcb_features.npz")
X2_train = data2["X_train"]
y2_train = data2["y_train"]
X2_test = data2["X_test"]
y2_test = data2["y_test"]

data = np.load("E:/Pythonfile/Facial-Recognition/data/feature/casia_lbp_face_u_hsv_features.npz")

print(X1_train.shape, X2_train.shape)

X_train = np.concatenate((X1_train, X2_train), axis=1)
y_train = y1_train
X_test = np.concatenate((X1_test, X2_test), axis=1)
y_test = y1_test

# X_train = data["X_train"]
# y_train = data["y_train"]
# X_test = data["X_test"]
# y_test = data["y_test"]

print(X_train.shape, X_test.shape)


(1010, 177) (1010, 177)
(1010, 354) (321, 354)


In [45]:

print("Features loaded!")
print("Training SVM...")

model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", C=10, gamma="scale", probability=True))
])

model.fit(X_train, y_train)

print("Evaluating...")
y_score = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

fpr, tpr, thresholds = roc_curve(y_test, y_score)

fnr = 1 - tpr

idx = np.nanargmin(np.abs(fnr - fpr))
eer = (fpr[idx] + fnr[idx]) / 2

print("EER:", eer)

Features loaded!
Training SVM...
Evaluating...
              precision    recall  f1-score   support

           0       0.96      1.00      0.98       240
           1       0.99      0.88      0.93        81

    accuracy                           0.97       321
   macro avg       0.97      0.94      0.95       321
weighted avg       0.97      0.97      0.97       321

EER: 0.04969135802469136


In [ ]:
print("Training XGBoost...")

model = Pipeline([
    ("scaler", StandardScaler()),   # không bắt buộc cho XGBoost nhưng giữ cũng không sao
    ("xgb", XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        use_label_encoder=False
    ))
])

model.fit(X_train, y_train)

print("Evaluating...")
y_score = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

fpr, tpr, thresholds = roc_curve(y_test, y_score)

fnr = 1 - tpr

idx = np.nanargmin(np.abs(fnr - fpr))
eer = (fpr[idx] + fnr[idx]) / 2

print("EER:", eer)


Training XGBoost...


e:\Pythonfile\Facial-Recognition\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [15:48:54] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Evaluating...
              precision    recall  f1-score   support

           0       0.90      0.97      0.94       240
           1       0.90      0.68      0.77        81

    accuracy                           0.90       321
   macro avg       0.90      0.83      0.86       321
weighted avg       0.90      0.90      0.90       321

EER: 0.1


In [11]:
from sklearn.ensemble import RandomForestClassifier

print("Training Random Forest...")

model = Pipeline([
    ("rf", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        n_jobs=-1,
        random_state=42
    ))
])
model.fit(X_train, y_train)

print("Evaluating...")
y_score = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

fpr, tpr, thresholds = roc_curve(y_test, y_score)

fnr = 1 - tpr

eer_threshold = thresholds[np.nanargmin(np.abs(fnr - fpr))]
eer = fpr[np.nanargmin(np.abs(fnr - fpr))]

print("EER:", eer)


Training Random Forest...
Evaluating...
              precision    recall  f1-score   support

           0       0.87      1.00      0.93       240
           1       1.00      0.57      0.72        81

    accuracy                           0.89       321
   macro avg       0.94      0.78      0.83       321
weighted avg       0.90      0.89      0.88       321

EER: 0.06666666666666667
